--- PREPARAZIONE DEI DATI ---

Pilastri della preparazione dei dati
Dalla verifica dei file alla gestione dello sbilanciamento
Per preparare i dati correttamente seguiamo 4 passaggi critici.
- 1) Integrità del formato: Scansionare dei file per identificare intestazioni corrotte (es. file JPEG troncati) che causerebbero errori durante la decodifica di TensorFlow.
- 2) Consistenza delle etichette: Verifica che ogni classe di animali (cani, gatti, uccelli) abbia un numero di campioni sufficiente e che non vi siano infiltrazioni di dati tra i set.
- 3) Normalizzazione spaziale: Ridimensionalmento uniforme di tutte le immagini, mantenendo se possibile il rapporto d'aspetto tramite tecniche di padding
- 4) Standarizzazione dei pixel: Trasformazione dei valori di intensità luminosa in un range numerico adatto alla convergenza del gradiente. Scaliamo i vettori da 0 a 255 per farli diventare a 0 a 1 in modo di aiutare la matematica del nostro gradiente a convergere

Gestire lo Sbilanciamento delle classi
Il mondo reale non è mai bilanciato, potrei avere migliaia di foto di un determinato cane, e poche foto di un raro gatto.
Possiamo usare oversempling, ovvero moltiplicare le foto rare, o il class weight che agiscono come un moltiplicatore di importanza.
- OverSampling e UnderSampling: Tecniche per equilibrare il numero di immagini per ogni razza di animale, evitando che il modello sviluppi un bias verso la classe più numerosa.
- Class Weights: Assegnazione di un peso maggiore alla loss del campione appartenenti a classi minoritarie durante la fase di ottimizzazione. Quando la rete sbaglia la classe rare, la punizione è molto più severa
- Filtraggio Outlier: Rimozione di immagini che non contengono l'animale target o che presentano un rumore eccessivo che confonderebbe il processo di estrazione delle feature.

Metriche di Bilanciamento
Il calcolo dei pesi per la funzione di perdita.
Per gestire dataset sbilanciati, calcoliamo i pesi basandoci sulla frequenza relativa della classi. Questo assicura che il contributo di ogni classe alla funzione di costo sia equo nonostante la disparità numerica.
L'approccio standard prevede l'utilizzo dell'inversa della frequenza di classe, spesso scalata per mantenere la magnitudo del gradiente sotto controllo.
E' un modo per dire alla rete di prestare massima attenzione a questi pochi esempi perchè sono 'preziosi'.

Ora che i dati sono puliti e pesati, dobbiamo portarli alla GPU


--- CARICARE I DATI NEL MODELLO ---

Setup della pipeline di training
Efficienza e scalabilità con tf.data
Una volta puliti i dati, dobbiamo caricarli nel modello in modo efficiente. Utilizzeremo l'API tf.data per costruire una pipeline asincrona che elimini i colli di bottiglia tra CPU e GPU.
La pipeline gestirà non solo il caricamento, ma anche l'augmentation in tempo reale, permettendo al modello di vedere variazioni diverse della stessa immagine di animale ed ogni epoca.

Componenti della pipeline
Ottimizzazione del flusso di dati
- 1) Map function: Applicazione di trasformazioni come il 'resinzing' e la 'normalization' su ogni elemento del dataset in modo parallelo.
- 2) Shuffle e Batch: Mescolamento dei dati ad ogni epoca per garantire la stocasticità e raggruppamento in batch per l'aggiornamento dei pesi.
- 3) Prefetching: Caricamente del batch successivo nella memoria della GPU mentre il modello sta ancora processando il batch corrente
- 4) Data Augmentation Integrata: Uso di layer Keras per rotazioni, flip e zoom che vengono eseguiti direttamente sulla GPU durante il training.

Ma possiamo spingerci oltre per rendere il tutto ancora più veloce.

Tecniche Avanzate di Pipeline
- 1) Chacing: salva i dati pre-processati in RAM o sul disco locale per evitare di ripetere operazioni costose ad ogni epoca di addestramento.
- 2) Parallelismo deterministico: Configurazione del parametro 'num_parallel_calls' impostato su 'tf.data.AUTOTUNE' per massimizzare l'uso dei core della CPU
- 3) Interleaving: Lettura simultanea da più file sorgenti per mitigare la latenza di input/output, particolarmente utile con dataset distribuiti su cloud. Legge da più dischi contemporaneamente nel caso di dataset sparsi.

Ma come misuriamo se stiamo andando davvero veloci?

Performance e Throughput
Il concetto di tempo di esecuzione del batch
Il tempo totale per un'epoca è dato dalla somma del tempo di caricamento dei dati e dal tempo di calcolo del gradiente. L'obbiettivo della pipeline è rendere il tempo di caricamente inferiore a quello di calcolo.
Vogliamo che il tempo totale dipenda solo dalla velocità della GPU. L'obbiettivo è arrivare ad avere una GPU con una % di utilizzo del 100%
Utilizzando il prefetch, sovrapponiamo le due fasi riducendo il tempo totale di idle della GPU

...

Siamo ora arrivati alla fase di deploy, abbiao addestrato il modello e dobbiamo esportarlo-

ESPORTAZIONE DEL MODELLO OTTIMIZZATO

Dal training alla produzione
Dopo aver addestrato il classificatore di animali, dobbiamo renderlo fruibile all'esterno. Dobbiamo salvare il modello in formato standard e ottimizzarlo per l'inferenza.
Ma un modello di Deep Learning può arrivare a pesare anche centinaio di MB, potrebbe essere necessario convertire il modello per dispositivi mobili, esempio APP che riconosce animali domestici.

Metodi di Salvataggio e Conversione
Formati e ottimizzazioni per il deploy
Abbiamo diverse opzioni:
- 1) SavedModel: il formato nativo di TensorFlow che include l'architettura, i pesi, il grafo, completo per il deploy tramite TF Serving
- 2) TF Lite Conversion: Trasformazione del modello in un formato flatbuffer leggero, ideale per smartphone e dispositivi edge
- 3) Quantizzazione: Riduzione della precisione dei pesi da float32 a Int8 per diminuire la dimensione del modello e accellelare l'inferenza senza perdere eccessiva accuratezza. E' come passare da un'immagine ad alta risoluzione ad una compressa ma ancora leggibile.
- 4) Pruning: Rimozione delle connessioni neuronali meno significative (pesi vicino a zero) per rendere il modello più compresso e veloce. Elimina le connessione che non servono.

Dettagli sull'esportazione
- Versioning: Organizzazione dei modelli salvati con tag di versione per permettere il rollback in caso di problemi in produzione
- Input Signatures: Definizione esplicita dei tipi e del range dei tensori di input attesi dal modello durante l'inferenza via API. Definiamo esattamente cosa accetta in ingresso.
- Metadata TFLite: inclusione di informazioni come nomi di classi e parametri di normalizzazione all'interno dei file .tflite per facilitare lo sviluppo mobile. Spiegano come gestire i dati in ingresso, esempio 0=cane.



In [2]:
"""
PIPELINE DI CLASSIFICAZIONE BINARIA (CAT VS DOG) - EDIZIONE 2025
Fix applicato: Risoluzione errore READ_VARIABLE nel convertitore TFLite.
"""

import os, pathlib, random, zipfile, requests
from PIL import Image

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ------------------------------------------------------------------
# 0. IMPOSTAZIONI GLOBALI & REPRODUCIBILITÀ
# ------------------------------------------------------------------
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE   = tf.data.AUTOTUNE
IMG_SIZE   = (160, 160)
BATCH_SIZE = 32

# ------------------------------------------------------------------
# 1. ACQUISIZIONE DATASET
# ------------------------------------------------------------------
DATA_URL = (
    "https://download.microsoft.com/download/"
    "3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
)

BASE_DIR  = pathlib.Path.home() / ".keras" / "datasets"
BASE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH   = BASE_DIR / "cats_and_dogs.zip"
EXTRACTED  = BASE_DIR / "PetImages"

if not ZIP_PATH.exists():
    print("[*] Scarico il dataset…")
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(DATA_URL, headers=headers)
    with open(ZIP_PATH, "wb") as f:
        f.write(r.content)

if not EXTRACTED.exists():
    print("[*] Estrazione in corso…")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(BASE_DIR)

DATA_DIR = BASE_DIR / "PetImages"

# ------------------------------------------------------------------
# 2. SANITIZZAZIONE DATI (Data Cleaning migliorata)
# ------------------------------------------------------------------
def clean_images(directory: pathlib.Path):
    """
    Rimuove file corrotti o con header JPEG non standard che causano i warning 
    'Corrupt JPEG data' e bloccano il training/conversione.
    """
    bad_files = 0
    for cls in ("Cat", "Dog"):
        folder = directory / cls
        for fname in os.listdir(folder):
            fpath = folder / fname
            if fpath.is_dir(): continue
            try:
                # La decodifica PIL è spesso più sensibile di tf.io alle corruzioni di byte
                with Image.open(fpath) as img:
                    img.verify() 
                # Ulteriore controllo via TF
                img_bytes = tf.io.read_file(str(fpath))
                _ = tf.image.decode_jpeg(img_bytes, channels=3)
            except Exception:
                bad_files += 1
                fpath.unlink()
    print(f"[+] {bad_files} file corrotti rimossi.")

clean_images(DATA_DIR)

# ------------------------------------------------------------------
# 3. PIPELINE DI INPUT (tf.data)
# ------------------------------------------------------------------
def get_datasets():
    raw_train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
    )

    raw_val_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
    )

    class_names = raw_train_ds.class_names

    # DATA AUGMENTATION: applichiamo solo durante il training, non durante la validazione
    # aiutano il modello a imparare l'inverianza spaziale (un cane rimane un cance anche se rotola)
    data_aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.05),
    ])

    def prepare(ds, augment=False):
        # Normalizzazione a [0, 1]
        ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)
        if augment:
            ds = ds.map(
                lambda x, y: (data_aug(x, training=True), y),
                num_parallel_calls=AUTOTUNE,
            )
        return ds.prefetch(AUTOTUNE)

    train_ds = prepare(raw_train_ds, augment=True)
    val_ds   = prepare(raw_val_ds, augment=False)

    return train_ds, val_ds, class_names

train_ds, val_ds, class_names = get_datasets()

# ------------------------------------------------------------------
# 4. ARCHITETTURA DEL MODELLO (CNN)
# ------------------------------------------------------------------
def build_model():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(), #per ridurre dimensionalità
        
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        
        layers.Flatten(), #trasformo in array semplici
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.5), 
        layers.Dense(1, activation="sigmoid"), #singolo neurone in uscita con attivazione sigmoide, perchè è una classificazione binaria
    ])
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model()
print("\n=== Architettura del modello ===")
model.summary()

# ------------------------------------------------------------------
# 5. STRATEGIE DI ADDESTRAMENTO
# ------------------------------------------------------------------
EPOCHS = 1 # Impostato a 1 per test rapido, aumentare in produzione

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

history = model.fit( #addestriamo con .fit
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
)

# ------------------------------------------------------------------
# 6. DIAGNOSTICA POST-TRAINING
# ------------------------------------------------------------------
#valutiamo accuracy per classe
def per_class_accuracy(model, val_ds):
    y_true, y_pred = [], []
    for x_batch, y_batch in val_ds:
        preds = model.predict(x_batch, verbose=0).flatten()
        y_true.extend(y_batch.numpy())
        y_pred.extend((preds > 0.5).astype(int))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print("\n--- Accuracy per classe ---")
    for idx, name in enumerate(class_names):
        mask = (y_true == idx)
        acc = np.mean(y_pred[mask] == y_true[mask])
        print(f"{name:>5}: {acc:.2%}")

per_class_accuracy(model, val_ds)

# ------------------------------------------------------------------
# 7. SALVATAGGIO E QUANTIZZAZIONE (Fix RuntimeError)
# ------------------------------------------------------------------
MODEL_DIR = "saved_model_pets"
tf.saved_model.save(model, MODEL_DIR)
print(f"\n[+] Modello salvato in {MODEL_DIR}")

def representative_data_gen():
    """
    Generatore per la calibrazione INT8. 
    Prendiamo i dati direttamente dal dataset già normalizzato.
    """
    # Prendiamo 100 campioni per la calibrazione
    for images, _ in train_ds.take(100):
        # I dati in train_ds sono già in batch di 32. 
        # TFLite richiede un sample alla volta o il batch intero? 
        # Iteriamo sul batch per sicurezza.
        for i in range(images.shape[0]):
            img = images[i:i+1] # Mantiene la forma (1, 160, 160, 3)
            yield [img.numpy()]

# FIX: Usiamo 'from_keras_model' invece di 'from_saved_model'.
# Questo risolve l'errore READ_VARIABLE perché il convertitore ha accesso
# diretto all'istanza Keras e alle sue variabili tracciate correttamente.
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Queste righe forzano il modello ad accettare INT8 in ingresso e uscita
# Utile per Edge TPU o microcontrollori.
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

try:
    tflite_quantized = converter.convert()
    with open("model_pets_quantized.tflite", "wb") as f:
        f.write(tflite_quantized)
    print("[+] Modello quantizzato TFLite scritto con successo!")
except Exception as e:
    print(f"[-] Errore durante la conversione: {e}")

c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


[+] 0 file corrotti rimossi.
Found 24824 files belonging to 2 classes.
Using 19860 files for training.
Found 24824 files belonging to 2 classes.
Using 4964 files for validation.

=== Architettura del modello ===


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 158, 158, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 79, 79, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 77, 77, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 38, 38, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 92416)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │     5,914,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,934,145 (22.64 MB)

...


[+] Modello salvato in saved_model_pets
INFO:tensorflow:Assets written to: C:\Users\uberti\AppData\Local\Temp\tmpf9r9m63p\assets


INFO:tensorflow:Assets written to: C:\Users\uberti\AppData\Local\Temp\tmpf9r9m63p\assets


Saved artifact at 'C:\Users\uberti\AppData\Local\Temp\tmpf9r9m63p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='keras_tensor_17')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2137530499152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676692432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676692048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676695504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676691088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676689744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676689552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2136676689936: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


[+] Modello quantizzato TFLite scritto con successo!
